# Chapter 02-04 · Missing values, duplicates, and impossible things

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** moderate - the taxonomy is easy, the judgement is not

**Prerequisites:** 02-03. You should be able to name the five selection mechanisms and say why more
data does not fix bias.

**Position in the learning path:** module 02, chapter 4 of 8. Before: **02-03**. After: **02-05**,
on distributions and outliers.

---

## Why this matters

The last chapter was about defects that leave no trace. This one is about the defects that do -
missing values, duplicates, impossible numbers, categories that mean the same thing - and the good
news is that they can all be *counted*.

The bad news is the first thing everybody does about them:

```python
df = df.dropna()
```

That line is a **selection mechanism**, and you have just spent a chapter learning what those do. In
this notebook it removes 22% of customers and shifts the average income by 13% - not because the
code is wrong, but because the people who declined to state their income were not a random 22%.

Every defect in this chapter needs a *decision*, and the decision depends on **why** the defect is
there - which is a provenance question, not a pandas question.

## What you will be able to do

By the end of this chapter you can:

1. **Distinguish** the three missingness mechanisms, and say which one makes dropping rows safe.
2. **Show** that mean imputation does not repair a biased complete-case analysis.
3. **Recognise** a sentinel value, and explain how one can invert a feature's meaning for a model.
4. **Find** exact and near-duplicates, impossible values and split categories, with code you can
   reuse.
5. **Choose** a handling strategy deliberately, and record the choice and the reason.

## Warm-up: retrieve, do not reread

From memory:

1. Name the five selection mechanisms.
2. Why does doubling a survey's responses not reduce its bias?
3. What direction does censoring always push a duration?
4. What does reweighting fix, and what can it never fix?

<br>

*Answers: (1) coverage, self-selection, survivorship, censoring, selection on the outcome. (2) more
data reduces variance, not bias - every extra response comes from the same skewed process.
(3) always understates it, because the long cases are still running. (4) differences on variables you
measured; never differences on variables you did not.*

## The situation

Maria's customer table has 3,000 rows. It also has holes: 22% of customers have no income recorded,
some rows appear twice, one bike is 200 years old, and the `station` column contains `north`,
`North` and `north ` as three separate values.

None of this is unusual. This *is* what data looks like.

**The question this chapter answers:** for each defect, what are you going to do about it - and what
does your choice assume?

## Three reasons a value is missing

The single most useful idea in this chapter is that missingness has a **mechanism**, and the
mechanism decides what you are allowed to do.

| Mechanism | Meaning | Maria's example | Is dropping rows safe? |
|---|---|---|---|
| **MCAR** - missing completely at random | The chance of being missing is unrelated to anything | A form was lost in transit | **Yes.** You lose precision, not correctness |
| **MAR** - missing at random *given what you observe* | Related to other recorded columns, not to the missing value itself | App users skip the income field; you record channel | **No** - but it can be repaired using the observed columns |
| **MNAR** - missing not at random | Related to the missing value itself | High earners decline to state their income | **No**, and nothing in the data can repair it |

The names are unhelpful - "missing at random" does not mean random - so hold on to the question
instead:

> **Does the chance of being missing depend on the value that is missing?**

If yes, you are in MNAR and the data cannot save you. If it depends only on things you recorded, you
are in MAR and a model can use them. If it depends on nothing, dropping rows is genuinely fine.

**And notice you cannot tell which you are in by looking.** The missing values are missing. Deciding
between MAR and MNAR is a judgement about the collection process - 02-02 again - not a calculation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(9)
n_customers = 3000

income = np.round(rng.lognormal(10.3, 0.5, n_customers), -2)          # SYNTHETIC, in EUR

# MNAR: customers above the 70th percentile are much likelier to decline.
p_decline = 0.05 + 0.55 * (income > np.quantile(income, 0.70))
declined = rng.random(n_customers) < p_decline

# MCAR: the same overall missing rate, but unrelated to income - a lost form.
lost_form = rng.random(n_customers) < declined.mean()

table = pd.DataFrame({
    "income": income,
    "income_mnar": np.where(declined, np.nan, income),
    "income_mcar": np.where(lost_form, np.nan, income),
})
print(f"true mean income: {income.mean():,.0f} EUR")
print(table[["income_mnar", "income_mcar"]].isna().mean().mul(100).round(1).to_string())

### Predict before running

Both columns are missing about 22% of their values.

1. Will `dropna()` on the MCAR column change the mean?
2. Will `dropna()` on the MNAR column change the mean, and in which direction?
3. Will filling the MNAR gaps with the *observed* mean repair the damage?

In [ ]:
true_mean = income.mean()
for name in ("income_mcar", "income_mnar"):
    complete_case = table[name].dropna().mean()
    print(f"{name}: complete-case mean {complete_case:8,.0f} EUR "
          f"({complete_case / true_mean - 1:+.1%} vs truth)")

filled = table["income_mnar"].fillna(table["income_mnar"].mean())
print(f"\nMNAR, gaps filled with the observed mean: {filled.mean():,.0f} EUR "
      f"({filled.mean() / true_mean - 1:+.1%})")

## Failure lab: `dropna()` is a selection mechanism

**MCAR: 33,951 against a true 33,941 - no bias at all.** Dropping 22% of rows cost precision and
nothing else. This is the case everyone imagines they are in.

**MNAR: 29,447 - 13.2% too low.** The customers who declined were the ones earning most, so removing
them removed the top of the distribution. Every downstream number inherits it: mean income, the
income-to-spending ratio, the customer segments, and any model that uses income as a feature.

**And filling the gaps with the observed mean gives exactly the same 29,447.**

That third result is the one worth sitting with. Mean imputation *looks* like a repair - the column
has no holes, the row count is restored, nothing warns you - and it changes the mean by precisely
nothing, because the mean it inserts is the biased mean. It converts a visible problem into an
invisible one.

It is worse than that. Mean imputation also:

- **shrinks the variance**, because a fifth of the column is now identical, so every measure of
  spread and every correlation involving income is distorted;
- **creates a spike** in the distribution at one value, which trees will happily split on;
- **destroys the information that the value was missing**, which - under MNAR - was itself
  informative. "Declined to answer" was telling you something about income, and you have thrown it
  away.

### What to do instead

| Approach | When it is right | What it assumes |
|---|---|---|
| **Drop the rows** | MCAR, and you can afford the rows | That missingness is unrelated to everything |
| **Add a "was missing" indicator column**, then impute | Almost always worth doing | Nothing. It lets a model learn that missingness itself is a signal |
| **Impute from other columns** (a model, or a group median) | MAR - the reason is recorded | That the observed columns explain the missingness |
| **Treat "missing" as its own category** | Categorical fields | That absence is a meaningful state, which it usually is |
| **Go and get the values** | When the field matters and the source exists | Effort, and it is the only true fix |
| **Report the analysis both ways** | When you cannot resolve it | Nothing, and it is honest |

**The indicator column is the highest-return line in this chapter.** One extra column,
`income_missing`, costs nothing and preserves a fact that every other approach discards. Under MNAR
it often turns out to be one of the most predictive features you have - which tells you something
important about the world, and would otherwise have been silently deleted.

In [ ]:
# The minimum responsible handling: keep the fact, then fill.
handled = table.assign(
    income_missing=table["income_mnar"].isna(),
    income_filled=table["income_mnar"].fillna(table["income_mnar"].median()),
)
print(handled.groupby("income_missing")["income"].agg(["size", "mean"]).round(0).to_string())
print("\nThe indicator is not noise: the two groups differ by "
      f"{handled.groupby('income_missing')['income'].mean().diff().iloc[-1]:,.0f} EUR in truth.")

The two groups differ by more than 20,000 EUR in true income - 29,447 against 50,157. That difference is exactly what the
indicator column carries, and exactly what `dropna()` and mean imputation throw away.

**A caution, so this is not read as a universal fix.** The indicator preserves the *signal* that a
value was missing; it does not recover the *value*. Your mean income is still wrong. And if
missingness in production has a different cause than in training - a form redesign, a new channel -
the indicator's meaning changes underneath the model. Record what the indicator means, in the data
dictionary, next to the column.

---

## Failure lab 2: the sentinel that inverted a feature

Maria's table has `days_since_last_rental`. For a customer who has **never** rented before, the
export writes **-1**, because the field cannot be empty and somebody needed a number.

This is a **sentinel value**: a real-looking number standing in for "not applicable" or "not
recorded". It loads as a valid number, `.isna()` finds nothing, and `.info()` reports a clean column.

**Predict before running:** a marketing rule targets customers who rented within the last 3 days.
What fraction of customers will it select, and who will they be?

In [ ]:
days = rng.integers(0, 60, n_customers).astype(float)
never_rented = rng.random(n_customers) < 0.18
recorded_days = np.where(never_rented, -1.0, days)                    # -1 means "never"

print(f"column dtype {recorded_days.dtype}, missing values found by isna(): "
      f"{np.isnan(recorded_days).sum()}")
print(f"mean of the column as loaded : {recorded_days.mean():.1f} days")
print(f"mean of the real values only : {days[~never_rented].mean():.1f} days")
print()
print(f"selected by 'rented in the last 3 days' : {(recorded_days <= 3).mean():.1%}")
print(f"actually rented in the last 3 days      : {((days <= 3) & ~never_rented).mean():.1%}")

### Diagnosis

**The rule selects 25.0% of customers. Only 5.7% genuinely qualify.**

The other 19.3 percentage points are people who have **never rented at all** - the exact opposite of
the group being targeted. Because -1 is less than 3, "never" sorts as "more recent than yesterday".

Nothing raised. `isna()` found zero missing values, because there are none: the column is a complete,
valid, numeric column that means something different from what its name says.

**Why models suffer worse than rules.** A rule at least has a threshold a human wrote and can
inspect. A model treats -1 as a point on the same scale as 0, 30 and 59, and learns a relationship
across a range that includes a value with no meaning. Worse, a *tree* will happily isolate -1 into
its own leaf and predict it well - looking like a useful split while encoding "we do not know".

**How to find sentinels.** They do not announce themselves, but they leave three tells:

1. **Impossible values.** A negative number of days, a birth year of 1900, an age of 999, a
   temperature of -273. Check the minimum and maximum of every numeric column against what the world
   allows.
2. **Suspicious spikes.** A histogram with a lonely tower at one value - 0, -1, 99, 9999. Real
   measurements are rarely that fond of round numbers.
3. **A value that appears far too often.** `value_counts().head()` on a numeric column takes one line
   and finds most of them.

In [ ]:
suspects = pd.Series(recorded_days).value_counts().head(3)
print("the three most common values in a supposedly continuous column:")
print(suspects.to_string())
print(f"\nminimum: {recorded_days.min()}  <- a negative number of days is impossible")

## The rest of the defects, and how to find them

Three more, each with the line of code that finds it.

In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 2, 3, 3, 4, 5, 6, 7],
    "station": ["north", "North", "North", "north ", "south", "south", "SOUTH", "east", "east"],
    "age": [34, 41, 41, 29, 29, 200, 38, -5, 52],
    "joined": pd.to_datetime(["2023-01-05", "2023-02-11", "2023-02-11", "2023-03-02", "2023-03-02",
                              "2023-04-19", "2023-05-30", "2023-06-14", "2099-01-01"]),
})

print("1. EXACT DUPLICATES")
print(customers[customers.duplicated(keep=False)].to_string(index=False))
print(f"   rows {len(customers)} -> unique {len(customers.drop_duplicates())}")

In [ ]:
print("2. DUPLICATE KEYS (the same id twice, rows not identical)")
print(customers[customers["customer_id"].duplicated(keep=False)].to_string(index=False))
print()
print("3. IMPOSSIBLE VALUES")
print(f"   ages outside 0-120 : {customers.loc[~customers['age'].between(0, 120), 'age'].tolist()}")
print(f"   join dates in the future: "
      f"{customers.loc[customers['joined'] > pd.Timestamp('2026-01-01'), 'joined'].dt.date.tolist()}")

In [ ]:
print("4. CATEGORIES THAT ARE THE SAME THING")
raw = customers["station"]
tidied = raw.str.strip().str.lower()
print(f"   distinct as recorded : {raw.nunique()}  {sorted(raw.unique())}")
print(f"   distinct after tidying: {tidied.nunique()}  {sorted(tidied.unique())}")
print()
print("   what a groupby would have reported:")
print(customers.groupby("station").size().to_string())

Look at that last block. `north`, `North` and `north ` are three separate groups holding four
customers between them, when they are one group of four. Any `groupby`, any one-hot encoding, any category-based split silently
treats them as unrelated - and a rare-category filter will drop all three as "too rare to model".

Notice how the `groupby` output prints two rows that look identical - `north` and `north `. The
trailing space is invisible on screen, which is exactly why this defect survives being looked at.

**The four checks, as four lines you can paste anywhere:**

```python
df.duplicated().sum()                                  # exact duplicates
df[key].duplicated().sum()                             # duplicate keys - see 01-05
df.describe()                                          # min/max against what the world allows
df[col].str.strip().str.lower().nunique() < df[col].nunique()   # categories that would merge
```

**One warning on duplicates**, because "drop them" is not always right. Two identical rows may be a
genuine repeat - two rentals of the same duration on the same day by the same person - and deleting
one destroys a real event. Whether an exact duplicate is a defect depends on whether the grain
allows repeats, which is 02-01's question. Check *why* before dropping.

## Common misconceptions

**"`dropna()` is the safe default."**
It is a selection filter, and it is safe only under MCAR - which you cannot verify from the data. It
is the most common way a careful analysis becomes biased.

**"Imputing with the mean is a neutral choice."**
It preserves the mean of the observed values, which is the wrong mean whenever missingness is
related to the value. It also shrinks variance, distorts correlations, and deletes the fact that the
value was missing.

**"`isna()` finds the missing data."**
It finds missing values that are represented as null. Sentinels, empty strings, `"N/A"` as text,
`"unknown"`, and 0 standing for "no data" are all invisible to it.

**"A complete column is a clean column."**
The sentinel column above was 100% complete and a quarter of it meant "not applicable".

**"Duplicates should always be dropped."**
Only if the grain forbids repeats. Ask what one row represents first.

**"Data cleaning comes before analysis."**
Cleaning *is* analysis. Every decision - drop, impute, cap, merge - changes what the data says, and
each one needs a reason recorded next to it. A cleaning step with no stated reason is an
undocumented modelling assumption.

**"I'll clean the data once and reuse it."**
Then the cleaning must be code, applied identically to training and to production data. A one-off
manual clean produces a training set that no future row will ever resemble - which is 13-04's
training/serving skew.

---

## Exercises

Solutions: `solutions/02_data_literacy/02-04_data_quality_solutions.ipynb`.

### Quick understanding

**E1 (define).** Define MCAR, MAR and MNAR in terms of one question, and say which makes `dropna()`
safe.

**E2 (explain).** Why did mean imputation leave the MNAR bias exactly unchanged?

**E3 (explain).** Why is a sentinel value more dangerous than a null?

### Hand calculation

**E4 (calculate).** Ten customers report incomes: 20, 22, 25, 28, 30, 34, 40, 55, 80, 120 (thousand
EUR). The three highest earners decline to answer. Compute the true mean, the complete-case mean, and
the mean after filling the three gaps with the observed mean. Then compute the true and complete-case
*medians*. Which summary held up better, and why?

**E5 (calculate).** A column is 25% sentinel `-1` values; the other 75% are real measurements
averaging 30, of which one in ten is 5 or less. Compute the reported mean of the column as loaded.
Then compute what share of rows a filter `value <= 5` selects, and what share it should select.

### Coding

**E6 (code).** Write `quality_report(df)` returning one row per column with: dtype, percentage
missing, number of distinct values, minimum and maximum for numeric columns, the most common value
and its share, and a flag for whether tidying whitespace and case would reduce the distinct count.

**E7 (code).** Write `find_sentinels(series, max_share=0.02)` that flags any value in a numeric
column appearing more often than `max_share` of rows, or lying outside a plausible range you pass
in. Run it on the chapter's `recorded_days`.

### Interpretation

**E8 (interpret).** Your `quality_report` shows a column that is 40% missing. Give three different
actions you might take, and the fact about the column that would decide between them.

### Debugging

**E9 (diagnose).** A model's accuracy is excellent in development and poor in production. In
development the team dropped rows with missing values; in production, missing values are filled with
zero by the serving code. Explain the two separate problems this creates and which one is worse.

### Exam and interview reasoning

**E10 (defend).** *"How do you handle missing data?"* Answer in about 130 words. A one-word answer
loses; so does listing five imputation methods.

**E11 (design).** You are given a dataset where 30% of `salary` values are missing, and you know from
the collection team that the field was optional for the first two years and mandatory afterwards.
Describe your approach, what you would check first, and what you would tell the person who asked for
"average salary".

### Transfer to a different situation

**E12 (design).** A hospital dataset has `blood_pressure` with 15% missing, `smoker` with the values
`Y`, `N`, `y`, `Yes`, `unknown` and blank, and `weight_kg` with a minimum of 0 and a maximum of 700.
For each column: name the likely mechanism or defect, the check you would run, and the handling you
would choose.

### Explain it to someone non-technical

**E13 (explain).** In under 70 words, explain to Maria why deleting the customers who left the income
field blank made her average income wrong, and what you did instead.

### Optional challenge

**E14 (code + diagnose).** Show the full downstream cost. Predict spending from income under three
treatments - drop rows, mean-impute, and impute-with-indicator - and compare each model's error
against a model fitted on the complete true data. Report which treatment recovers most of the lost
performance and explain why.

In [ ]:
# Your workspace. Still in memory: table, income, declined, lost_form, handled,
# recorded_days, days, never_rented, customers.

## Mastery check

Without scrolling up, can you:

- [ ] State the one question that separates MCAR, MAR and MNAR? *(If not: "Three reasons".)*
- [ ] Say why mean imputation did not help? *(If not: "Failure lab".)*
- [ ] Explain how -1 inverted a feature's meaning? *(If not: "Failure lab 2".)*
- [ ] Name the four one-line checks for duplicates, keys, impossible values and categories?
      *(If not: "The rest of the defects".)*
- [ ] Say why an indicator column is worth adding even when you impute? *(If not: "What to do
      instead".)*

## What should now feel instinctive

1. **"Why is this missing?"** - before deciding what to do about it.
2. **`dropna()` is a filter.** Say what it removed and check whether the removed rows differ.
3. **Add the indicator column.** One line, keeps a fact every other approach deletes.
4. **Check `min` and `max` against what the world allows** - the cheapest defect detector there is.
5. **Every cleaning decision gets a recorded reason.** Cleaning is modelling.

## Flashcards

| Question | Answer |
|---|---|
| The one missingness question | Does the chance of being missing depend on the value that is missing? |
| MCAR, MAR, MNAR | Unrelated to anything; related to observed columns; related to the missing value itself |
| When is `dropna()` safe? | MCAR only - and you cannot verify MCAR from the data |
| Why doesn't mean imputation help under MNAR? | It inserts the biased observed mean, so the mean is unchanged - and variance shrinks |
| What does an indicator column preserve? | The fact that the value was missing, which is often informative |
| What is a sentinel? | A real-looking number standing in for "not recorded" - `-1`, `0`, `999` |
| Why are sentinels worse than nulls? | `isna()` cannot see them, and they sort and average as if they were data |
| Three tells of a sentinel | Impossible values; a spike in the histogram; one value appearing far too often |
| Should duplicates always be dropped? | Only if the grain forbids repeats - check what one row represents |
| The cheapest defect check | `describe()`, comparing min and max against what the world allows |

## Next

**02-05 · Distributions, outliers, and transformations.**

You can now find and handle the values that are wrong. The next chapter is about values that are
*right* and surprising - the long tail, the skew, the single enormous day - and the question that
separates a good analyst from a careless one: is this outlier an error to remove, or the most
important row in the dataset?

New terms are in [GLOSSARY.md](../../GLOSSARY.md).